<a href="https://colab.research.google.com/github/mariya7040ai/-arabic-english-medqa-uncertainty/blob/main/Another_copy_of_Paper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
from google.colab import userdata
from huggingface_hub import login, whoami
login(userdata.get("HF_TOKEN"))
print("logged in as:", whoami()["name"])

logged in as: mariyaaleem


In [29]:
from google.colab import userdata
from huggingface_hub import login, hf_hub_download
login(userdata.get("HF_TOKEN"))
p = hf_hub_download("inception42/jais-adapted-7b-chat", "config.json")
print("access OK:", p)

access OK: /root/.cache/huggingface/hub/models--inception42--jais-adapted-7b-chat/snapshots/f2de64b06baedc5546928fbdea10fca517f7cbc7/config.json


In [7]:
# 1. Install and imports
!pip -q install -U bitsandbytes accelerate transformers

In [8]:
"""# 1b. CONFIG: edit only this cell
LANG = "en"                                   # "en" = English (MedMCQA), "ar" = Arabic (MedAraBench)
MODEL_NAME =  """

'# 1b. CONFIG: edit only this cell\nLANG = "en"                                   # "en" = English (MedMCQA), "ar" = Arabic (MedAraBench)\nMODEL_NAME =  '

In [9]:
"""# 1b. CONFIG: edit only this cell
LANG = "en"                                   # "en" = English (MedMCQA), "ar" = Arabic (MedAraBench)
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"       # later try the Arabic-focused model, for example inceptionai/jais-adapted-7b-chat
MEDARA_CSV = "/content/medarabench_test.csv"  # only used when LANG = "ar"
N_TEST = 50                                   # Day 2 test size
N_FULL = 3000                                 # Day 3 full-run size
SEED = 42"""

'# 1b. CONFIG: edit only this cell\nLANG = "en"                                   # "en" = English (MedMCQA), "ar" = Arabic (MedAraBench)\nMODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"       # later try the Arabic-focused model, for example inceptionai/jais-adapted-7b-chat\nMEDARA_CSV = "/content/medarabench_test.csv"  # only used when LANG = "ar"\nN_TEST = 50                                   # Day 2 test size\nN_FULL = 3000                                 # Day 3 full-run size\nSEED = 42'

In [10]:
# 1b. CONFIG: edit only this cell
LANG = "en"                                   # "en" = English (MedMCQA), "ar" = Arabic (MedAraBench)
MODEL_NAME = "inception42/jais-adapted-7b-chat"       # later try the Arabic-focused model, for example inceptionai/jais-adapted-7b-chat
MEDARA_CSV = "/content/drive/MyDrive/medqa_data/medarabench-test_split.csv"   # use the exact path you printed
N_TEST = 50                                   # Day 2 test size
N_FULL = 5000                                 # Day 3 full-run size
SEED = 42

In [11]:
"""# 1b. CONFIG: edit only this cell
LANG = "ar"                                   # "en" = English (MedMCQA), "ar" = Arabic (MedAraBench)
#MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"       # later try the Arabic-focused model, for example inceptionai/jais-adapted-7b-chat
MODEL_NAME = "inceptionai/jais-adapted-7b-chat"
MEDARA_CSV = "/content/medarabench_test.csv"  # only used when LANG = "ar"
N_TEST = 50                                   # Day 2 test size
N_FULL = 1000                                 # Day 3 full-run size
SEED = 42"""

'# 1b. CONFIG: edit only this cell\nLANG = "ar"                                   # "en" = English (MedMCQA), "ar" = Arabic (MedAraBench)\n#MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"       # later try the Arabic-focused model, for example inceptionai/jais-adapted-7b-chat\nMODEL_NAME = "inceptionai/jais-adapted-7b-chat"\nMEDARA_CSV = "/content/medarabench_test.csv"  # only used when LANG = "ar"\nN_TEST = 50                                   # Day 2 test size\nN_FULL = 1000                                 # Day 3 full-run size\nSEED = 42'

In [12]:
# 2. Seeds and logging
import os, json, time, random, datetime
import numpy as np, pandas as pd, torch

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

LETTERS = list("ABCDEF")

def log_run(text, path="RUN_LOG.md"):
    ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    with open(path, "a", encoding="utf-8") as f:
        f.write(f"- {ts}: {text}\n")
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

torch 2.11.0+cu130 | cuda: True


In [13]:
# 3. Functions to load MedAraBench (columns as in your Day 1 output)
def load_medarabench(path):
    df = pd.read_csv(path)
    opt_cols = [f"Option {l}" for l in LETTERS]
    rows, bad = [], {"empty_question": 0, "answer_not_a_letter": 0,
                     "answer_points_to_empty_option": 0, "fewer_than_2_options": 0}
    for i, r in df.iterrows():
        q = r["Question"]
        if not isinstance(q, str) or not q.strip():
            bad["empty_question"] += 1; continue
        opts, letters = [], []
        for l, c in zip(LETTERS, opt_cols):
            v = r[c]
            if isinstance(v, str) and v.strip():
                opts.append(v.strip()); letters.append(l)
        ans = r["Correct Answer"]
        if not isinstance(ans, str) or ans.strip().upper() not in LETTERS:
            bad["answer_not_a_letter"] += 1; continue
        ans = ans.strip().upper()
        if ans not in letters:
            bad["answer_points_to_empty_option"] += 1; continue
        if len(opts) < 2:
            bad["fewer_than_2_options"] += 1; continue
        rows.append(dict(id=int(i), question=q.strip(), options=opts,
                         answer_idx=letters.index(ans),
                         specialty=str(r["umbrella_specialty"]), level=str(r["Level"])))
    return pd.DataFrame(rows), bad

In [14]:
# 4. Helpers: convert a table to the common format, and sample questions
def df_to_schema(df, q_col, opt_cols, ans_col, answer_is_index=False, specialty_col=None, level_col=None):
    rows = []
    for i, r in df.iterrows():
        opts = [str(r[c]).strip() for c in opt_cols if isinstance(r[c], str) and str(r[c]).strip()]
        a = r[ans_col]
        idx = int(a) if answer_is_index else LETTERS.index(str(a).strip().upper())
        if len(opts) < 2 or idx >= len(opts):
            continue
        rows.append(dict(id=int(i), question=str(r[q_col]).strip(), options=opts, answer_idx=idx,
                         specialty=str(r[specialty_col]) if specialty_col else "NA",
                         level=str(r[level_col]) if level_col else "NA"))
    return pd.DataFrame(rows)

def sample_df(df, n, seed=SEED):
    return df.sample(n=min(n, len(df)), random_state=seed).reset_index(drop=True)

In [15]:
"""# 4b. Load the data for the language chosen in CONFIG
if LANG == "ar":
    data_df, bad = load_medarabench(MEDARA_CSV)
    print("dropped:", bad)
else:
    from datasets import load_dataset
    en_raw = load_dataset("openlifescienceai/medmcqa", split="validation").to_pandas()
    print(en_raw.shape, en_raw.columns.tolist())
    print("cop values:\n", en_raw["cop"].value_counts())
    if "choice_type" in en_raw.columns:
        print("choice_type values:\n", en_raw["choice_type"].value_counts())
    spec = "subject_name" if "subject_name" in en_raw.columns else None
    data_df = df_to_schema(en_raw[en_raw["cop"] >= 0], "question", ["opa", "opb", "opc", "opd"], "cop",
                           answer_is_index=True, specialty_col=spec)
K = data_df["options"].map(len)
print(LANG, "usable questions:", len(data_df))
print("options per question:\n", K.value_counts().sort_index())
print("chance accuracy (mean of 1/K): %.3f" % (1 / K).mean())
print("answer position distribution:\n", data_df["answer_idx"].value_counts(normalize=True).sort_index().round(3))
print("specialties:\n", data_df["specialty"].value_counts().head(25))
log_run(f"Loaded {LANG} data: {len(data_df)} usable questions, chance acc {(1/K).mean():.3f}")"""

'# 4b. Load the data for the language chosen in CONFIG\nif LANG == "ar":\n    data_df, bad = load_medarabench(MEDARA_CSV)\n    print("dropped:", bad)\nelse:\n    from datasets import load_dataset\n    en_raw = load_dataset("openlifescienceai/medmcqa", split="validation").to_pandas()\n    print(en_raw.shape, en_raw.columns.tolist())\n    print("cop values:\n", en_raw["cop"].value_counts())\n    if "choice_type" in en_raw.columns:\n        print("choice_type values:\n", en_raw["choice_type"].value_counts())\n    spec = "subject_name" if "subject_name" in en_raw.columns else None\n    data_df = df_to_schema(en_raw[en_raw["cop"] >= 0], "question", ["opa", "opb", "opc", "opd"], "cop",\n                           answer_is_index=True, specialty_col=spec)\nK = data_df["options"].map(len)\nprint(LANG, "usable questions:", len(data_df))\nprint("options per question:\n", K.value_counts().sort_index())\nprint("chance accuracy (mean of 1/K): %.3f" % (1 / K).mean())\nprint("answer position distribu

In [16]:
# 4b. Load the data for the language chosen in CONFIG
ENGLISH_SINGLE_ONLY = True    # True = keep only choice_type == "single" questions (recommended); False = keep all

if LANG == "ar":
    data_df, bad = load_medarabench(MEDARA_CSV)
    print("dropped:", bad)
else:
    from datasets import load_dataset
    en_raw = load_dataset("openlifescienceai/medmcqa", split="validation").to_pandas()
    print(en_raw.shape, en_raw.columns.tolist())
    print("cop values:\n", en_raw["cop"].value_counts())
    if "choice_type" in en_raw.columns:
        print("choice_type values:\n", en_raw["choice_type"].value_counts())
    keep = en_raw[en_raw["cop"] >= 0]
    if ENGLISH_SINGLE_ONLY and "choice_type" in keep.columns:
        keep = keep[keep["choice_type"] == "single"]
        print("kept single-choice questions only:", len(keep))
    spec = "subject_name" if "subject_name" in en_raw.columns else None
    data_df = df_to_schema(keep, "question", ["opa", "opb", "opc", "opd"], "cop",
                           answer_is_index=True, specialty_col=spec)
K = data_df["options"].map(len)
print(LANG, "usable questions:", len(data_df))
print("options per question:\n", K.value_counts().sort_index())
print("chance accuracy (mean of 1/K): %.3f" % (1 / K).mean())
print("answer position distribution:\n", data_df["answer_idx"].value_counts(normalize=True).sort_index().round(3))
print("specialties:\n", data_df["specialty"].value_counts().head(25))
log_run(f"Loaded {LANG} data (single-only={ENGLISH_SINGLE_ONLY if LANG == 'en' else 'n/a'}): {len(data_df)} usable questions, chance acc {(1/K).mean():.3f}")

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 85.9MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  936kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/182822 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6150 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4183 [00:00<?, ? examples/s]

(4183, 11) ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name']
cop values:
 cop
0    1348
1    1085
2     925
3     825
Name: count, dtype: int64
choice_type values:
 choice_type
single    2816
multi     1367
Name: count, dtype: int64
kept single-choice questions only: 2816
en usable questions: 2816
options per question:
 options
4    2816
Name: count, dtype: int64
chance accuracy (mean of 1/K): 0.250
answer position distribution:
 answer_idx
0    0.337
1    0.261
2    0.219
3    0.183
Name: proportion, dtype: float64
specialties:
 specialty
Dental                          845
Pathology                       259
Surgery                         249
Medicine                        185
Pharmacology                    179
Gynaecology & Obstetrics        154
Anatomy                         147
Pediatrics                      133
Physiology                      133
Biochemistry                    122
Social & Preventive Medicine     91
Mi

In [17]:
# 5. Load ONE model in 4-bit (MODEL_NAME comes from the CONFIG cell). Gated models need a Hugging Face token in Colab Secrets (HF_TOKEN).
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata; from huggingface_hub import login; login(userdata.get("HF_TOKEN"))

MODEL_TAG = MODEL_NAME.split("/")[-1]

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb, device_map="auto")
model.eval()
print("loaded", MODEL_NAME, "| GPU memory used (GB): %.1f" % (torch.cuda.memory_allocated() / 1e9))
log_run(f"Loaded {MODEL_NAME} in 4-bit on {torch.cuda.get_device_name(0)}")

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.36k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B / 1.18MB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/4.37M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

loaded inception42/jais-adapted-7b-chat | GPU memory used (GB): 5.8


In [18]:
# 6. Prompt and option probabilities (one forward pass per question, no text generation)
INSTRUCTION = "Answer the following medical multiple-choice question with the letter of the correct option only.\n\n"

PROMPT_STYLE = "no_prefill"   # options: "no_prefill", "answer_colon", "answer_is" (ignored for Jais, which uses its own template)
SUFFIXES = {"no_prefill": "", "answer_colon": "Answer:", "answer_is": "The correct answer is"}

IS_JAIS = "jais" in MODEL_NAME.lower()
USE_CHAT_TEMPLATE = bool(tok.chat_template) and not IS_JAIS
JAIS_SYSTEM = "Your name is Jais. You are a helpful, respectful and honest assistant made by Inception. Complete the conversation between [|Human|] and [|AI|]:"

def build_prompt(question, options):
    body = question + "\n" + "\n".join(f"{l}. {o}" for l, o in zip(LETTERS, options))
    user = INSTRUCTION + body
    if IS_JAIS:
        return f"### Instruction:{JAIS_SYSTEM}\n### Input: [|Human|] {user}\n[|AI|]\n### Response :"
    if USE_CHAT_TEMPLATE:
        text = tok.apply_chat_template([{"role": "user", "content": user}], tokenize=False, add_generation_prompt=True)
    else:
        text = user + "\n"
    return text + SUFFIXES[PROMPT_STYLE]

def letter_ids(letters):
    out = []
    for l in letters:
        c = set()
        for s in (l, " " + l):
            t = tok.encode(s, add_special_tokens=False)
            if t: c.add(t[-1])
        out.append(sorted(c))
    return out
LETTER_IDS = letter_ids(LETTERS)

@torch.no_grad()
def option_probs(question, options):
    text = build_prompt(question, options)
    enc = tok(text, return_tensors="pt", add_special_tokens=not USE_CHAT_TEMPLATE).to(model.device)
    logp = torch.log_softmax(model(**enc).logits[0, -1].float(), dim=-1)
    scores = torch.stack([torch.logsumexp(logp[LETTER_IDS[k]], dim=0) for k in range(len(options))])
    mass = torch.exp(torch.logsumexp(scores, dim=0)).item()      # probability mass the model puts on valid letters
    probs = torch.softmax(scores, dim=0)
    return probs.cpu().numpy(), scores.cpu().numpy(), mass

In [ ]:
"""PROMPT_STYLE = "no_prefill"   # options: "no_prefill", "answer_colon", "answer_is"
SUFFIXES = {"no_prefill": "", "answer_colon": "Answer:", "answer_is": "The correct answer is"}

def build_prompt(question, options):
    body = question + "\n" + "\n".join(f"{l}. {o}" for l, o in zip(LETTERS, options))
    user = INSTRUCTION + body
    if tok.chat_template:
        text = tok.apply_chat_template([{"role": "user", "content": user}], tokenize=False, add_generation_prompt=True)
    else:
        text = user + "\n"
    return text + SUFFIXES[PROMPT_STYLE]"""

In [19]:
# 7. Sanity check on 3 questions
test = sample_df(data_df, 3)
for _, r in test.iterrows():
    p, s, m = option_probs(r["question"], r["options"])
    print("K =", len(r["options"]), "| probs:", np.round(p, 3), "| mass on letters: %.2f" % m,
          "| pred:", LETTERS[int(p.argmax())], "| true:", LETTERS[int(r["answer_idx"])])

K = 4 | probs: [0.302 0.138 0.235 0.325] | mass on letters: 0.93 | pred: D | true: A
K = 4 | probs: [0.071 0.277 0.416 0.237] | mass on letters: 0.76 | pred: C | true: B
K = 4 | probs: [0.084 0.13  0.5   0.286] | mass on letters: 0.94 | pred: C | true: C


In [20]:
# 8. Evaluation loop with saving and resume (safe against Colab disconnects)
from tqdm.auto import tqdm

def run_eval(df, tag, out_dir="results"):
    os.makedirs(out_dir, exist_ok=True)
    path = f"{out_dir}/{tag}.jsonl"
    done = set()
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            done = {json.loads(l)["id"] for l in f}
    t0, n_new = time.time(), 0
    with open(path, "a", encoding="utf-8") as f:
        for _, r in tqdm(df.iterrows(), total=len(df)):
            if int(r["id"]) in done:
                continue
            probs, scores, mass = option_probs(r["question"], r["options"])
            rec = dict(id=int(r["id"]), K=len(r["options"]), answer=int(r["answer_idx"]),
                       pred=int(np.argmax(probs)), conf=float(probs.max()), mass=float(mass),
                       probs=[float(x) for x in probs], scores=[float(x) for x in scores],
                       specialty=str(r["specialty"]), level=str(r["level"]))
            f.write(json.dumps(rec, ensure_ascii=False) + "\n"); n_new += 1
            if n_new % 25 == 0:
                f.flush()
    secs = time.time() - t0
    print(f"{n_new} new questions in {secs/60:.1f} min ({secs/max(n_new,1):.2f} s per question)")
    return path

# Day 2 test run (N_TEST questions)
sub = sample_df(data_df, N_TEST)
path = run_eval(sub, f"{MODEL_TAG}_{LANG}_seed{SEED}_n{N_TEST}")

  0%|          | 0/50 [00:00<?, ?it/s]

50 new questions in 0.3 min (0.31 s per question)


In [21]:
# 9. Day 2 diagnostics: is the pipeline sane?
def load_results(path):
    with open(path, encoding="utf-8") as f:
        return pd.DataFrame([json.loads(l) for l in f])

res = load_results(path)
res["correct"] = res["pred"] == res["answer"]
print("accuracy: %.3f | chance: %.3f" % (res["correct"].mean(), (1 / res["K"]).mean()))
print("mean mass on valid letters: %.3f (should be close to 1; if low, the prompt format needs fixing)" % res["mass"].mean())
print("predicted position distribution:\n", res["pred"].value_counts(normalize=True).sort_index().round(2))
print("true position distribution:\n", res["answer"].value_counts(normalize=True).sort_index().round(2))
log_run(f"Day 2 test {MODEL_TAG} {LANG} n={len(res)}: acc={res['correct'].mean():.3f}, mass={res['mass'].mean():.3f}")

accuracy: 0.400 | chance: 0.250
mean mass on valid letters: 0.900 (should be close to 1; if low, the prompt format needs fixing)
predicted position distribution:
 pred
0    0.10
1    0.24
2    0.46
3    0.20
Name: proportion, dtype: float64
true position distribution:
 answer
0    0.26
1    0.18
2    0.32
3    0.24
Name: proportion, dtype: float64


In [22]:
print(res["mass"].describe().round(3))
print("questions with mass < 0.9:", int((res["mass"] < 0.9).sum()), "of", len(res))
print(res.assign(low=res["mass"] < 0.9).groupby("low")["correct"].agg(["mean", "count"]).round(3))

count    50.000
mean      0.900
std       0.041
min       0.763
25%       0.890
50%       0.904
75%       0.929
max       0.965
Name: mass, dtype: float64
questions with mass < 0.9: 20 of 50
       mean  count
low               
False  0.50     30
True   0.25     20


In [23]:
def mass_test(style, df=sub):
    global PROMPT_STYLE
    PROMPT_STYLE = style
    ms, cs = [], []
    for _, r in df.iterrows():
        p, s, m = option_probs(r["question"], r["options"])
        ms.append(m); cs.append(int(p.argmax()) == int(r["answer_idx"]))
    ms = np.array(ms)
    print(f"{style:13s} mean mass {ms.mean():.3f} | median {np.median(ms):.3f} | share < 0.9: {(ms < 0.9).mean():.2f} | acc {np.mean(cs):.3f}")

for st in ["no_prefill", "answer_colon", "answer_is"]:
    mass_test(st)

no_prefill    mean mass 0.900 | median 0.904 | share < 0.9: 0.40 | acc 0.400
answer_colon  mean mass 0.900 | median 0.904 | share < 0.9: 0.40 | acc 0.400
answer_is     mean mass 0.900 | median 0.904 | share < 0.9: 0.40 | acc 0.400


In [24]:
PROMPT_STYLE = "no_prefill"
print("prompt style:", PROMPT_STYLE)

prompt style: no_prefill


In [25]:
# 10. Calibration metrics
from scipy.optimize import minimize_scalar
import matplotlib.pyplot as plt

def softmax(x):
    x = np.asarray(x, dtype=float); x = x - x.max(); e = np.exp(x); return e / e.sum()

def ece(conf, correct, n_bins=15):
    conf, correct = np.asarray(conf), np.asarray(correct, dtype=float)
    idx = np.digitize(conf, np.linspace(0, 1, n_bins + 1)[1:-1])
    tot = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            tot += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(tot)

def brier(df):
    v = []
    for p, a in zip(df["probs"], df["answer"]):
        p = np.asarray(p); oh = np.zeros_like(p); oh[a] = 1.0
        v.append(((p - oh) ** 2).sum())
    return float(np.mean(v))

def boot_ci(df, metric, B=1000, seed=0, level=0.95):
    rng = np.random.default_rng(seed); n = len(df); vals = []
    for _ in range(B):
        vals.append(metric(df.iloc[rng.integers(0, n, n)]))
    lo, hi = np.percentile(vals, [100 * (1 - level) / 2, 100 * (1 + level) / 2])
    return float(lo), float(hi)

def metric_acc(d): return float((d["pred"] == d["answer"]).mean())
def metric_ece(d): return ece(d["conf"].values, (d["pred"] == d["answer"]).values)

def fit_temperature(scores_list, y):
    def nll(logT):
        T = np.exp(logT)
        return -np.mean([np.log(softmax(np.asarray(s) / T)[yi] + 1e-12) for s, yi in zip(scores_list, y)])
    r = minimize_scalar(nll, bounds=(-3, 3), method="bounded")
    return float(np.exp(r.x))

def reliability_plot(conf, correct, n_bins=10, title=""):
    conf, correct = np.asarray(conf), np.asarray(correct, dtype=float)
    idx = np.digitize(conf, np.linspace(0, 1, n_bins + 1)[1:-1]); xs, ys = [], []
    for b in range(n_bins):
        m = idx == b
        if m.any():
            xs.append(conf[m].mean()); ys.append(correct[m].mean())
    plt.figure(figsize=(4.5, 4.5)); plt.plot([0, 1], [0, 1], "--", color="gray"); plt.plot(xs, ys, "o-")
    plt.xlabel("Mean confidence"); plt.ylabel("Accuracy"); plt.title(title); plt.xlim(0, 1); plt.ylim(0, 1); plt.show()

In [26]:
for name in ["MODEL_TAG", "tok", "model", "option_probs", "run_eval", "load_results", "ece", "data_df"]:
    print(f"{name:14s}", name in globals())

MODEL_TAG      True
tok            True
model          True
option_probs   True
run_eval       True
load_results   True
ece            True
data_df        True


In [27]:
for name in ["MODEL_TAG", "tok", "model", "option_probs", "run_eval", "load_results", "ece", "data_df"]:
    print(f"{name:14s}", name in globals())
print("LANG =", LANG, "| N_FULL =", N_FULL, "| usable questions:", len(data_df))

MODEL_TAG      True
tok            True
model          True
option_probs   True
run_eval       True
load_results   True
ece            True
data_df        True
LANG = en | N_FULL = 5000 | usable questions: 2816


In [28]:
# 11. Full run
PROMPT_STYLE = "no_prefill"
sub = sample_df(data_df, N_FULL)
N = len(sub)

"""# Ensure MODEL_TAG is defined
if "MODEL_TAG" not in globals() and "MODEL_NAME" in globals() and MODEL_NAME:
    MODEL_TAG = MODEL_NAME.split("/")[-1]
else:
    MODEL_TAG = "model"
"""
tag = f"{MODEL_TAG}_{LANG}_{PROMPT_STYLE}_seed{SEED}_n{N}"
path = run_eval(sub, tag)
log_run(f"Day 3 full run finished: {tag}")

  0%|          | 0/2816 [00:00<?, ?it/s]

2816 new questions in 15.0 min (0.32 s per question)


In [ ]:
print(path)

In [ ]:
import os
MODEL_TAG = MODEL_NAME.split("/")[-1]
old = path
new = f"results/{MODEL_TAG}_{LANG}_{PROMPT_STYLE}_seed{SEED}_n{N}.jsonl"
if os.path.exists(old) and old != new:
    os.rename(old, new)
path = new
tag = f"{MODEL_TAG}_{LANG}_{PROMPT_STYLE}_seed{SEED}_n{N}"
print(path, os.path.exists(path))
log_run(f"Renamed results file to {path} (the earlier tag used a placeholder model name)")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/medqa_results
!cp -r results RUN_LOG.md /content/drive/MyDrive/medqa_results/
print("backed up")

In [ ]:
import os
good = "/content/drive/MyDrive/medqa_results/results/Qwen2.5-7B-Instruct_ar_no_prefill_seed42_n4927.jsonl"
bad = "/content/drive/MyDrive/medqa_results/results/model_ar_no_prefill_seed42_n4927.jsonl"
if os.path.exists(good) and os.path.exists(bad):
    os.remove(bad); print("removed the wrongly named copy")
else:
    print("good copy exists:", os.path.exists(good), "| bad copy exists:", os.path.exists(bad))

In [ ]:
# 12. Accuracy, ECE, Brier with 95% bootstrap confidence intervals
res = load_results(path)
print("n =", len(res), "| chance acc: %.3f" % (1 / res["K"]).mean(), "| mass on letters: %.3f" % res["mass"].mean())
for name, fn in [("accuracy", metric_acc), ("ECE (15 bins)", metric_ece), ("Brier", brier)]:
    val = fn(res); lo, hi = boot_ci(res, fn)
    print(f"{name}: {val:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")
reliability_plot(res["conf"].values, (res["pred"] == res["answer"]).values, title=f"{MODEL_TAG}, {LANG} (raw)")
log_run(f"{tag}: acc={metric_acc(res):.4f}, ECE={metric_ece(res):.4f}, Brier={brier(res):.4f}")

In [ ]:
rng = np.random.default_rng(0)
perm = rng.permutation(len(res)); half = len(res)//2
cal, te = res.iloc[perm[:half]], res.iloc[perm[half:]]
T = fit_temperature(list(cal["scores"]), cal["answer"].values)
conf_ts = np.array([softmax(np.asarray(s)/T).max() for s in te["scores"]])
corr = (te["pred"].values == te["answer"].values)

def rel_points(conf, correct, n_bins=10):
    idx = np.digitize(conf, np.linspace(0,1,n_bins+1)[1:-1]); xs, ys = [], []
    for b in range(n_bins):
        m = idx == b
        if m.any(): xs.append(conf[m].mean()); ys.append(correct[m].mean())
    return xs, ys

plt.figure(figsize=(4.8,4.8)); plt.plot([0,1],[0,1],"--",color="gray")
plt.plot(*rel_points(te["conf"].values, corr), "o-", label="raw")
plt.plot(*rel_points(conf_ts, corr), "s-", label=f"temperature scaled (T={T:.1f})")
plt.xlabel("Mean confidence"); plt.ylabel("Accuracy"); plt.title(f"{MODEL_TAG}, {LANG}")
plt.legend(); plt.xlim(0,1); plt.ylim(0,1); plt.show()
print("T =", round(T, 2), "| ECE raw", round(ece(te["conf"].values, corr), 3), "-> TS", round(ece(conf_ts, corr), 3))

In [ ]:
print("mean confidence: %.3f | accuracy: %.3f" % (res["conf"].mean(), metric_acc(res)))
hi = res["conf"] > 0.99
print("share of questions with confidence > 0.99: %.3f" % hi.mean())
print("accuracy on those questions: %.3f" % (res.loc[hi, "pred"] == res.loc[hi, "answer"]).mean())

In [ ]:
# 13. Temperature scaling with repeated random 50/50 splits (fit T on calibration half, evaluate on test half)
def ts_experiment(df, R=20, seed=0):
    rng = np.random.default_rng(seed); out = []
    for _ in range(R):
        perm = rng.permutation(len(df)); half = len(df) // 2
        cal, te = df.iloc[perm[:half]], df.iloc[perm[half:]]
        T = fit_temperature(list(cal["scores"]), cal["answer"].values)
        p2 = [softmax(np.asarray(s) / T) for s in te["scores"]]
        conf2 = np.array([p.max() for p in p2]); corr = (te["pred"].values == te["answer"].values)
        out.append(dict(T=T, ece_raw=ece(te["conf"].values, corr), ece_ts=ece(conf2, corr)))
    return pd.DataFrame(out)

ts = ts_experiment(res, R=20)
print(ts.describe().loc[["mean", "std"]].round(4))
log_run(f"{tag}: temperature scaling over 20 splits: mean T={ts['T'].mean():.3f}, ECE raw={ts['ece_raw'].mean():.4f} -> {ts['ece_ts'].mean():.4f}")

In [ ]:
# 14. Per-specialty table (only specialties with at least MIN_N questions)
MIN_N = 30
rows = []
for sp, g in res.groupby("specialty"):
    if len(g) >= MIN_N:
        rows.append(dict(specialty=sp, n=len(g), accuracy=metric_acc(g), ECE=metric_ece(g), mean_conf=g["conf"].mean()))
tbl = pd.DataFrame(rows).sort_values("n", ascending=False).round(3)
tbl

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp -r /content/drive/MyDrive/medqa_results/results .
!cp /content/drive/MyDrive/medqa_results/RUN_LOG.md .

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/medqa_results
!cp -r results RUN_LOG.md /content/drive/MyDrive/medqa_results/
print("backed up")